# Predicting late delivery

Question: can we predict `late_delivery` at dispatch time? Each consignment appears once, and a model would be used on later consignments, so the test set is the **latest rows** by `dispatched_on`.

Feature policy (from `columns.csv`): `consignment` is an identifier, dropped. `claim_paid_eur` is recorded after delivery (a claim settled for the consignment), so it leaks the outcome and is dropped. Features known at dispatch: `carrier`, `distance_km`, `weight_kg`, `parcels`, and time-of-dispatch features derived from `dispatched_on`.

In [ ]:
import pandas as pd, numpy as np
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score

df = pd.read_csv('data.csv', parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
df['hour'] = df.dispatched_on.dt.hour
df['dow'] = df.dispatched_on.dt.dayofweek
print(df.shape, df.dispatched_on.min(), df.dispatched_on.max(), df.late_delivery.mean())

## 1. Hold out the test set (latest 20% of rows by time)

In [ ]:
cut = int(len(df)*0.8)
train, test = df.iloc[:cut].copy(), df.iloc[cut:].copy()
print(len(train), len(test), train.dispatched_on.max(), test.dispatched_on.min())
FEATS = ['carrier','distance_km','weight_kg','parcels','hour','dow']
NUM = ['distance_km','weight_kg','parcels','hour','dow']
Xtr, ytr = train[FEATS], train.late_delivery
Xte, yte = test[FEATS], test.late_delivery
# test set is not touched again until the final cell

## 2. Candidate models (named before any scoring)

1. `baseline`: predicts the training prevalence (trivial).
2. `logreg`: standardised numeric + one-hot carrier, logistic regression (C=1).
3. `logreg_c01`: same with C=0.1.
4. `hgb`: histogram gradient boosting (learning rate 0.05, max_depth 3, 150 iterations).

Selection metric: mean ROC AUC over expanding-window time-series CV inside the training rows. The decision threshold is chosen on out-of-fold predictions from CV, maximising F1. Preprocessing lives inside each pipeline, so it is fit within folds.

In [ ]:
def prep():
    return ColumnTransformer([('num', StandardScaler(), NUM), ('cat', OneHotEncoder(handle_unknown='ignore'), ['carrier'])])
models = {
 'baseline': Pipeline([('p',prep()),('m',DummyClassifier(strategy='prior'))]),
 'logreg': Pipeline([('p',prep()),('m',LogisticRegression(C=1,max_iter=1000))]),
 'logreg_c01': Pipeline([('p',prep()),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'hgb': Pipeline([('p',prep()),('m',HistGradientBoostingClassifier(learning_rate=0.05,max_depth=3,max_iter=150,random_state=0))]),
}

## 3. Time-ordered cross-validation on training rows only

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
oof = {k: np.full(len(train), np.nan) for k in models}
rows=[]
for k,m in models.items():
    aucs=[];aps=[]
    for tr,va in tscv.split(Xtr):
        f = clone(m).fit(Xtr.iloc[tr], ytr.iloc[tr])
        p = f.predict_proba(Xtr.iloc[va])[:,1]
        oof[k][va]=p
        aucs.append(roc_auc_score(ytr.iloc[va],p)); aps.append(average_precision_score(ytr.iloc[va],p))
    rows.append((k,np.mean(aucs),np.std(aucs),np.mean(aps)))
cv = pd.DataFrame(rows,columns=['model','cv_auc','auc_sd','cv_ap']).set_index('model'); cv

## 4. Choose model and threshold on CV only

In [ ]:
best = cv.cv_auc.drop('baseline').idxmax()
mask = ~np.isnan(oof[best])
grid = np.linspace(0.05,0.8,76)
f1s = [f1_score(ytr[mask], oof[best][mask]>=t) for t in grid]
thr = float(grid[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(thr,3), '| CV F1:', round(max(f1s),3))
print('CV baseline AUC:', cv.loc['baseline','cv_auc'].round(3))

**Decision (recorded before test scoring):** the model, features and threshold printed above are final. Features: carrier, distance_km, weight_kg, parcels, hour, dow. Nothing will be changed after seeing test results.

## 5. Refit on all training rows; score once on the test set

In [ ]:
final = clone(models[best]).fit(Xtr, ytr)
base = clone(models['baseline']).fit(Xtr, ytr)
p = final.predict_proba(Xte)[:,1]; pb = base.predict_proba(Xte)[:,1]
pred = p>=thr
print(f'TEST ROC AUC {roc_auc_score(yte,p):.3f} (baseline {roc_auc_score(yte,pb):.3f})')
print(f'TEST avg precision {average_precision_score(yte,p):.3f} (prevalence {yte.mean():.3f})')
print(f'At threshold {thr:.2f}: precision {precision_score(yte,pred):.3f}, recall {recall_score(yte,pred):.3f}, F1 {f1_score(yte,pred):.3f}')

## 6. Diagnostic after the headline (descriptive only; no choices changed)

The test prevalence is far below the training prevalence, so the late rate over time is inspected.

In [ ]:
print(df.groupby(df.dispatched_on.dt.to_period('M')).late_delivery.agg(['size','mean']).round(3))
print('train prevalence', round(ytr.mean(),3), '| test prevalence', round(yte.mean(),3))

## Conclusion
The headline is the single test-set score in section 5. Dispatch-time features carry modest ranking signal (CV AUC about 0.72 vs 0.5 for the baseline; coastal carrier and longer distances run later). On the held-out latest rows the AUC is about 0.70, but the late rate collapses from roughly 20-30% to 1-5% from October 2025, so the threshold chosen on CV (F1-optimal for ~25% prevalence) gives very low precision on the test period. The model ranks consignments reasonably but its calibration and threshold do not transfer to the new regime; it would need recalibration, and the cause of the drop (process change?) should be investigated. `claim_paid_eur` was excluded as post-outcome leakage.